# Attention Sink Mechanism Plotting Notebook

This notebook contains plotting and analysis code for *The Structural Origin of Attention Sink: Variance Discrepancy, Super Neurons, and Dimension Disparity* (ICML 2026).

Model paths are configured through environment variables instead of local absolute paths:

```bash
export LLAMA2_MODEL_PATH=/path/to/Llama-2-7b-hf
export LLAMA3_MODEL_PATH=/path/to/Llama-3-8B
export DEVICE=cuda:0
```

If these variables are not set, the notebook falls back to Hugging Face model identifiers.


## Variance Discrepancy Across Positions

Aligns with Sec. 3.1 and Fig. 4. Measures the position-wise standard deviation caused by causal value aggregation.


In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from tqdm import tqdm
import gc
import os

# =========================================
# =========================================
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 18,
    'axes.labelsize': 20,
    'axes.titlesize': 22,
    'xtick.labelsize': 16,
    'ytick.labelsize': 16,
    'figure.titlesize': 24,
    'lines.linewidth': 3,
    'lines.markersize': 8,
    'grid.alpha': 0.4,
    'mathtext.fontset': 'stix',
})

MODEL_PATH = os.getenv("LLAMA2_MODEL_PATH", "meta-llama/Llama-2-7b-hf")
DEVICE = os.getenv("DEVICE", "cuda:0" if torch.cuda.is_available() else "cpu") 
SEQ_LEN = 64
NUM_SAMPLES = 20
TARGET_LAYERS = [0, 1, 2]

# =========================================
# =========================================
def prepare_and_inspect_data(tokenizer, num_samples, seq_len):
    print("Loading WikiText-2 dataset...")
    data = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
    long_text = [x['text'] for x in data if len(x['text']) > seq_len * 4]
    
    indices = np.random.choice(len(long_text), num_samples, replace=False)
    
    samples_tokens = []
    
    print("\n" + "="*50)
    print(f"Inspecting {num_samples} Sampled Sequences (Length {seq_len})")
    print("="*50)
    
    for i, idx in enumerate(indices):
        txt = long_text[idx]
        tokens = tokenizer(txt, return_tensors="pt").input_ids
        
        if tokens.shape[1] >= seq_len:
            tokens = tokens[:, :seq_len]
            samples_tokens.append(tokens)
            
            if i < 3: 
                decoded = tokenizer.decode(tokens[0])
                print(f"[Sample {i+1}] (First 100 chars): {decoded[:100]}...")
                print("-" * 50)
    
    print(f"Total valid samples collected: {len(samples_tokens)}")
    return samples_tokens

# =========================================
# =========================================
def analyze_std_evolution(model_path, device, samples):
    print(f"\nLoading model model from {model_path}...")
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        device_map=device,
        torch_dtype=torch.float16,
        trust_remote_code=True
    )
    model.eval()

    layer_stats = {l: np.zeros((NUM_SAMPLES, SEQ_LEN)) for l in TARGET_LAYERS}
    
    def get_std_hook(layer_idx, sample_idx_tracker):
        def hook(module, args):
            hidden_states = args[0].detach().float()
            
            stds = hidden_states.std(dim=-1).squeeze(0).cpu().numpy()
            
            current_sample_idx = sample_idx_tracker[0]
            if current_sample_idx < NUM_SAMPLES:
                layer_stats[layer_idx][current_sample_idx, :] = stds
                
        return hook

    handles = []
    sample_tracker = [0] 
    
    print(f"Registering hooks on Layers {TARGET_LAYERS}...")
    for l in TARGET_LAYERS:
        if hasattr(model, 'model') and hasattr(model.model, 'layers'):
            target_module = model.model.layers[l].self_attn.o_proj
        elif hasattr(model, 'transformer') and hasattr(model.transformer, 'h'):
            target_module = model.transformer.h[l].attn.c_proj
        else:
             target_module = model.model.layers[l].self_attn.o_proj

        h = target_module.register_forward_pre_hook(get_std_hook(l, sample_tracker))
        handles.append(h)

    print("Running inference...")
    with torch.no_grad():
        for inp in tqdm(samples):
            inp = inp.to(device)
            model(inp)
            sample_tracker[0] += 1

    for h in handles: h.remove()
    del model
    gc.collect()
    torch.cuda.empty_cache()

    final_results = {}
    for l in TARGET_LAYERS:
        # Mean across samples
        mean_curve = layer_stats[l].mean(axis=0)
        # Std across samples (for error bars if needed, though usually just mean curve is clean)
        final_results[l] = mean_curve
        
    return final_results

# =========================================
# =========================================
def plot_std_evolution(results):
    fig, ax = plt.subplots(figsize=(12, 8), constrained_layout=True)
    
    colors = sns.color_palette("viridis", len(results))
    
    x_axis = np.arange(SEQ_LEN)
    
    for i, layer_idx in enumerate(sorted(results.keys())):
        y_data = results[layer_idx]
        
        ax.plot(x_axis, y_data, 
                color=colors[i], 
                marker='o', 
                markersize=6, 
                label=f'Layer {layer_idx}',
                alpha=0.9)
        
        ax.scatter([0], [y_data[0]], color=colors[i], s=150, edgecolors='black', zorder=10)

    ax.set_title(f"Llama: Representation Variance Evolution (Input to $W_O$)\n(Averaged over {NUM_SAMPLES} samples)", 
                 fontsize=22, fontweight='bold', pad=20)
    
    ax.set_xlabel("Token Position Index", fontsize=20)
    ax.set_ylabel(r"Standard Deviation $\sigma(h)$", fontsize=20)
    
    ax.annotate('First Token Anomaly\n(High Variance)', 
                xy=(0, results[0][0]), 
                xytext=(5, results[0][0]),
                arrowprops=dict(facecolor='black', arrowstyle='->', lw=2),
                fontsize=16, fontweight='bold')

    ax.legend(fontsize=16, loc='center right')
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.set_xlim(-1, SEQ_LEN)
    
    plt.show()

if __name__ == "__main__":
    print(f"Initializing Tokenizer from {MODEL_PATH}...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
    
    samples = prepare_and_inspect_data(tokenizer, NUM_SAMPLES, SEQ_LEN)
    
    results = analyze_std_evolution(MODEL_PATH, DEVICE, samples)
    
    plot_std_evolution(results)

## Mask Intervention

Aligns with Sec. 3.2 and Fig. 5 / Appendix Fig. 19. Blocks aggregation at a chosen token to induce an attention sink away from position 0.


In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import matplotlib.patches as patches
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from tqdm import tqdm
import gc
import os

# =========================================
# =========================================
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 34,
    'axes.labelsize': 36,
    'axes.titlesize': 38,
    'xtick.labelsize': 32,
    'ytick.labelsize': 32,
    'figure.titlesize': 40,
})

MODELS_CONFIG = [
    {"name": "Llama-2-7b", "path": os.getenv("LLAMA2_MODEL_PATH", "meta-llama/Llama-2-7b-hf")},
    {"name": "Llama-3-8b", "path": os.getenv("LLAMA3_MODEL_PATH", "meta-llama/Meta-Llama-3-8B")}
]

SEQ_LEN = 64
NUM_SAMPLES = 20
ISOLATED_TOKEN_IDX = 9
DEVICE = os.getenv("DEVICE", "cuda:0" if torch.cuda.is_available() else "cpu") 

# =========================================
# =========================================
def get_wikitext_samples(tokenizer, num_samples, seq_len):
    samples = []
    try:
        data = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
        texts = [x['text'] for x in data if len(x['text']) > seq_len * 4]
        indices = np.random.permutation(len(texts))
        for idx in indices:
            if len(samples) >= num_samples: break
            tokens = tokenizer(texts[idx], return_tensors="pt").input_ids
            if tokens.shape[1] >= seq_len:
                samples.append(tokens[:, :seq_len])
    except: pass
    if len(samples) < num_samples:
        while len(samples) < num_samples:
            samples.append(torch.randint(0, 32000, (1, seq_len)))
    return samples

def get_averaged_attention_patterns(model_path, seq_len, isolated_idx, device):
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_path, device_map=device, torch_dtype=torch.float16,
        attn_implementation="eager", trust_remote_code=True
    ).eval()

    num_layers = model.config.num_hidden_layers
    layers_to_plot = [0, num_layers // 2, num_layers - 1]
    layer_labels = ["First Layer", "Middle Layer", "Last Layer"]
    samples = get_wikitext_samples(tokenizer, NUM_SAMPLES, SEQ_LEN)
    accumulated_maps = {idx: np.zeros((seq_len, seq_len)) for idx in layers_to_plot}
    current_batch_maps = {}

    def hook(module, input, output):
        current_batch_maps[module.layer_idx] = output[1].mean(dim=1).squeeze(0).detach().float().cpu().numpy()

    hooks = []
    layers_obj = model.model.layers if hasattr(model, "model") else model.layers
    for i in layers_to_plot:
        layer = layers_obj[i]
        layer.self_attn.layer_idx = i
        hooks.append(layer.self_attn.register_forward_hook(hook))

    mask = torch.full((1, 1, seq_len, seq_len), torch.finfo(torch.float16).min, device=device, dtype=torch.float16)
    mask.masked_fill_(torch.tril(torch.ones((seq_len, seq_len), device=device, dtype=torch.bool)).unsqueeze(0).unsqueeze(0), 0.0)
    mask[:, :, isolated_idx, :isolated_idx] = torch.finfo(torch.float16).min

    with torch.no_grad():
        for inp in tqdm(samples):
            current_batch_maps.clear()
            model(inp.to(device), attention_mask=mask)
            for idx in layers_to_plot:
                accumulated_maps[idx] += current_batch_maps[idx]

    results = [(idx, layer_labels[i], accumulated_maps[idx]/len(samples)) for i, idx in enumerate(layers_to_plot)]
    for h in hooks: h.remove()
    del model, tokenizer
    torch.cuda.empty_cache()
    gc.collect()
    return results

# =========================================
# =========================================

def plot_single_model(model_name, results):
    fig, axes = plt.subplots(1, 3, figsize=(24, 10), sharex=True, sharey=True, constrained_layout=True)
    
    ticks = [0] + list(range(9, SEQ_LEN, 10))
    tick_labels = [str(t + 1) for t in ticks]

    for col_idx, (layer_idx, label, data) in enumerate(results):
        ax = axes[col_idx]
        
        # Heatmap
        sns.heatmap(data, ax=ax, cmap="viridis", cbar=True, square=True, cbar_kws={"shrink": 0.7})
        
        ax.set_xticks([t + 0.5 for t in ticks])
        ax.set_xticklabels(tick_labels, rotation=0)
        ax.set_yticks([t + 0.5 for t in ticks])
        ax.set_yticklabels(tick_labels, rotation=0)

        ax.set_title(f"{label} (Layer {layer_idx+1})", pad=25)
        ax.set_xlabel("Key Position")
        if col_idx == 0: 
            ax.set_ylabel("Query Position")

        rect = patches.Rectangle((0, ISOLATED_TOKEN_IDX), SEQ_LEN, 1, 
                                 linewidth=4.0, edgecolor='red', facecolor='none', zorder=10)
        ax.add_patch(rect)
        
        ax.text(SEQ_LEN / 2, ISOLATED_TOKEN_IDX - 2.5, f"Isolated Token {ISOLATED_TOKEN_IDX + 1}", 
                color='red', fontsize=30, fontweight='bold', ha='center', va='center',
                bbox=dict(facecolor='white', alpha=0.85, edgecolor='red', boxstyle='round,pad=0.2'),
                zorder=11)

    fig.suptitle(f"[{model_name}] Average Attention Pattern (Intervention at Token {ISOLATED_TOKEN_IDX+1})", 
                 fontweight='bold', y=1.02)
    plt.show()

if __name__ == "__main__":
    for config in MODELS_CONFIG:
        print(f"\nProcessing {config['name']}...")
        res = get_averaged_attention_patterns(config["path"], SEQ_LEN, ISOLATED_TOKEN_IDX, DEVICE)
        plot_single_model(config["name"], res)

## Variance Amplification

Aligns with Sec. 3.2 and Fig. 6 / Appendix Fig. 20. Applies mean-centered variance amplification to a target token and measures the induced sink score.


In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import gc
import os

# =========================================
# =========================================
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 22,
    'axes.labelsize': 24,
    'axes.titlesize': 26,
    'xtick.labelsize': 18,
    'ytick.labelsize': 18,
    'legend.fontsize': 20,
    'figure.titlesize': 28,
    'lines.linewidth': 3,
    'lines.markersize': 10,
    'grid.alpha': 0.4,
})

MODELS_CONFIG = {
    "Llama-2-7b": {
        "path": os.getenv("LLAMA2_MODEL_PATH", "meta-llama/Llama-2-7b-hf"),
        "color": "#1f77b4", 
        "marker": "o",
        "target_layer": 2   
    },
    "Llama-3-8b": {
        "path": os.getenv("LLAMA3_MODEL_PATH", "meta-llama/Meta-Llama-3-8B"),
        "color": "#d62728", 
        "marker": "s",
        "target_layer": 2   
    }
}

DEVICE = os.getenv("DEVICE", "cuda:0" if torch.cuda.is_available() else "cpu")
SEQ_LEN = 64
NUM_SAMPLES = 200     
TARGET_Token_IDX = 10  
SCALES_TO_TEST = [1.0, 2.0, 5.0, 10.0, 15.0, 20.0, 25.0, 30.0]

class MultiLayerAmplificationExp:
    def __init__(self, model_path, device, target_layer):
        print(f"Loading Model from {model_path}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
            
        self.model = AutoModelForCausalLM.from_pretrained(
            model_path, 
            device_map=device, 
            torch_dtype=torch.float16, 
            attn_implementation="eager", 
            trust_remote_code=True
        ).eval()
        self.device = device
        self.target_layer = target_layer
        self.hidden_size = self.model.config.hidden_size
        self.layer_means = {} 
        self.num_q_heads = self.model.config.num_attention_heads
        self.num_kv_heads = getattr(self.model.config, "num_key_value_heads", self.num_q_heads)
        self.head_dim = self.hidden_size // self.num_q_heads
        self.gqa_group_size = self.num_q_heads // self.num_kv_heads

    def phase1_compute_all_layer_means(self, num_tokens=50000):
        print(f"  > Phase 1: Computing Layer-wise Means (Random Baseline)...")
        aggregators = {l: None for l in range(self.target_layer)}
        counts = {l: 0 for l in range(self.target_layer)}
        
        def get_hook(layer_idx):
            def hook(module, args, output):
                o_val = output.detach().float() 
                if aggregators[layer_idx] is None:
                    aggregators[layer_idx] = torch.zeros(o_val.shape[-1], device=self.device, dtype=torch.float32)
                aggregators[layer_idx].add_(o_val.sum(dim=(0, 1)))
                counts[layer_idx] += o_val.shape[0] * o_val.shape[1]
            return hook

        handles = []
        for l in range(self.target_layer):
            h = self.model.model.layers[l].self_attn.v_proj.register_forward_hook(get_hook(l))
            handles.append(h)
        
        batch_size = 32
        vocab_size = self.model.config.vocab_size
        num_batches = max(1, num_tokens // (batch_size * SEQ_LEN) + 1)
        with torch.no_grad():
            for _ in range(num_batches):
                inp = torch.randint(0, vocab_size, (batch_size, SEQ_LEN)).to(self.device)
                self.model(inp)
            
        for h in handles: h.remove()
        for l in range(self.target_layer):
            self.layer_means[l] = aggregators[l] / counts[l]

    def _expand_mean_if_needed(self, mu, target_dim):
        current_dim = mu.shape[0]
        if current_dim == target_dim: return mu
        if self.gqa_group_size > 1 and current_dim * self.gqa_group_size == target_dim:
            mu = mu.view(self.num_kv_heads, self.head_dim)
            mu = mu[:, None, :].expand(self.num_kv_heads, self.gqa_group_size, self.head_dim)
            return mu.reshape(-1)
        else:
            raise ValueError(f"Dim mismatch: {current_dim} vs {target_dim}")

    def _get_amplification_hook(self, layer_idx, scale):
        mu_raw = self.layer_means[layer_idx].to(self.device).float()
        def hook(module, args):
            o_states = args[0]
            target_vec = o_states[:, TARGET_Token_IDX, :].float()
            mu = self._expand_mean_if_needed(mu_raw, target_vec.shape[-1])
            delta = target_vec - mu
            new_vec = mu + scale * delta
            o_states[:, TARGET_Token_IDX, :] = new_vec.to(o_states.dtype)
            return (o_states,)
        return hook

    def phase2_run_experiment(self):
        print(f"  > Phase 2: Running Cumulative Amplification...")
        try:
            dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
        except Exception as e:
            print(f"    Error: {e}"); return []
        real_inputs = []
        for text in dataset['text']:
            if len(text) < 100: continue
            tokens = self.tokenizer(text, return_tensors="pt", add_special_tokens=False).input_ids
            if tokens.shape[1] >= SEQ_LEN:
                real_inputs.append(tokens[:, :SEQ_LEN])
            if len(real_inputs) >= NUM_SAMPLES: break
        
        results = []
        for scale in SCALES_TO_TEST:
            intervention_handles = []
            attn_handle = None
            try:
                for l in range(self.target_layer):
                    h = self.model.model.layers[l].self_attn.o_proj.register_forward_pre_hook(
                        self._get_amplification_hook(l, scale)
                    )
                    intervention_handles.append(h)
                attn_scores = []
                def attn_check(module, input, output):
                    if output[1] is None: return
                    attn = output[1].detach().float().cpu()
                    score = attn[:, :, (TARGET_Token_IDX+1):, TARGET_Token_IDX].mean().item()
                    attn_scores.append(score)
                attn_handle = self.model.model.layers[self.target_layer].self_attn.register_forward_hook(attn_check)
                with torch.no_grad():
                    batch_size = 20
                    for i in range(0, len(real_inputs), batch_size):
                        batch_list = real_inputs[i:i+batch_size]
                        if not batch_list: break
                        batch = torch.cat(batch_list, dim=0).to(self.device)
                        self.model(batch, output_attentions=True)
                avg_score = np.mean(attn_scores)
                results.append(avg_score)
                print(f"    Lambda {scale:>4.1f}: Score = {avg_score:.4f}")
            finally:
                for h in intervention_handles: h.remove()
                if attn_handle: attn_handle.remove()
        return results

def plot_comparative_results(all_results):
    fig, ax = plt.subplots(figsize=(14, 8))
    for model_name, res_data in all_results.items():
        scores = res_data['scores']
        cfg = MODELS_CONFIG[model_name]
        ax.plot(SCALES_TO_TEST, scores, color=cfg['color'], marker=cfg['marker'], 
                linewidth=3, markersize=10, label=f"{model_name}(Obs: L3)")
        
        if 1.0 in SCALES_TO_TEST:
            idx_1 = SCALES_TO_TEST.index(1.0)
            xytext_offset = (3.0, 0.15) if "Llama-2" in model_name else (3.0, 0.4)
            ax.annotate(f'Natural ({model_name})', 
                        xy=(1.0, scores[idx_1]), 
                        xytext=(1.0 + xytext_offset[0], scores[idx_1] + xytext_offset[1]),
                        arrowprops=dict(facecolor=cfg['color'], arrowstyle='->'),
                        fontsize=16, color=cfg['color'])

    ax.set_title("Sink Induction via Cumulative Amplification", fontweight='bold', pad=20)
    ax.set_xlabel(r"Amplification Factor $\lambda$ (Applied to Preceding Layers)")
    ax.set_ylabel(f"Attention Score to Token {TARGET_Token_IDX}")
    ax.set_ylim(0, 1.05)
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(loc='lower right')
    
    ax.text(0.05, 0.95, r"For $l < L_{obs}: \mathbf{o}^{(l)} \leftarrow \boldsymbol{\mu}^{(l)} + \lambda(\mathbf{o}^{(l)} - \boldsymbol{\mu}^{(l)})$", 
            transform=ax.transAxes, fontsize=18, va='top',
            bbox=dict(facecolor='wheat', alpha=0.5))
    plt.show()

if __name__ == "__main__":
    all_experiment_results = {}
    gc.collect(); torch.cuda.empty_cache()
    for model_name, cfg in MODELS_CONFIG.items():
        print(f"\n=== Processing {model_name} ===")
        try:
            exp = MultiLayerAmplificationExp(cfg['path'], DEVICE, cfg['target_layer'])
            exp.phase1_compute_all_layer_means()
            scores = exp.phase2_run_experiment()
            all_experiment_results[model_name] = {"scores": scores}
            del exp; gc.collect(); torch.cuda.empty_cache()
        except Exception as e:
            print(f"Error: {e}")
    if all_experiment_results:
        plot_comparative_results(all_experiment_results)

## Output Projection Preservation

Aligns with Sec. 4.1 and Fig. 7. Measures whether the attention output projection preserves the first-token variance discrepancy.


In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from tqdm import tqdm
import gc
import os

# =========================================
# =========================================
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 18,
    'axes.labelsize': 20,
    'axes.titlesize': 22,
    'xtick.labelsize': 16,
    'ytick.labelsize': 16,
    'figure.titlesize': 24,
    'lines.linewidth': 3,
    'lines.markersize': 10,
    'grid.alpha': 0.4,
})

CONFIGS = [
    {
        "name": "Llama-2-7b",
        "path": os.getenv("LLAMA2_MODEL_PATH", "meta-llama/Llama-2-7b-hf"),
        "target_layer": 1,
        "color": "#1f77b4"
    }
]

SEQ_LEN = 64
NUM_SAMPLES = 20
DEVICE = os.getenv("DEVICE", "cuda:0" if torch.cuda.is_available() else "cpu") 

# =========================================
# =========================================
def get_wikitext_samples(tokenizer, num_samples, seq_len):
    """  WikiText-2  """
    print(f"Loading WikiText-2 samples (Len: {seq_len})...")
    samples = []
    try:
        data = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
        texts = [x['text'] for x in data if len(x['text']) > seq_len * 4]
        indices = np.random.permutation(len(texts))
        
        for idx in indices:
            if len(samples) >= num_samples: break
            tokens = tokenizer(texts[idx], return_tensors="pt").input_ids
            if tokens.shape[1] >= seq_len:
                samples.append(tokens[:, :seq_len])
                
    except Exception as e:
        print(f"Warning: Data load failed ({e}). Using dummy data.")

    if len(samples) < num_samples:
        while len(samples) < num_samples:
            vocab_size = getattr(tokenizer, "vocab_size", 32000)
            samples.append(torch.randint(0, vocab_size, (1, seq_len)))
            
    return samples

# =========================================
# =========================================

def get_avg_oproj_input_std(config, seq_len, num_samples, device):
    model_path = config["path"]
    layer_idx = config["target_layer"]
    
    print(f"Loading {config['name']} from {model_path}...")
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        device_map=device,
        torch_dtype=torch.float16,
        attn_implementation="eager", 
        trust_remote_code=True
    )
    model.eval()

    samples = get_wikitext_samples(tokenizer, num_samples, seq_len)
    
    current_std_container = {"val": None}

    def hook_fn(module, args, output):
        x = args[0]
        token_std = x.std(dim=-1).detach().float().cpu().numpy()
        current_std_container["val"] = token_std[0]

    if hasattr(model, "model") and hasattr(model.model, "layers"):
        target_module = model.model.layers[layer_idx].self_attn.o_proj
    elif hasattr(model, "layers"): 
        target_module = model.layers[layer_idx].self_attn.o_proj
    else:
        raise ValueError("Unknown model structure")

    handle = target_module.register_forward_hook(hook_fn)

    accumulated_std = np.zeros(seq_len)
    count = 0

    print(f" > Running inference to capture Layer {layer_idx} O-Proj Input Std...")
    
    with torch.no_grad():
        for inp in tqdm(samples):
            inp = inp.to(device)
            model(inp)
            
            if current_std_container["val"] is not None:
                accumulated_std += current_std_container["val"]
                count += 1
            
            current_std_container["val"] = None

    handle.remove()
    del model, tokenizer
    torch.cuda.empty_cache()
    gc.collect()

    avg_std = accumulated_std / max(count, 1)
    return avg_std

# =========================================
# =========================================

def plot_std_analysis_single(results):
    """
     (Averaged Data)
    """
    config, std_data = results[0]
    
    name = config['name']
    layer = config['target_layer']
    color = config['color']

    fig, ax = plt.subplots(figsize=(14, 8), constrained_layout=True)
    
    ax.plot(np.arange(SEQ_LEN), std_data, 
            color=color, 
            marker='o',
            markeredgecolor='white',
            markeredgewidth=1.5,
            linestyle='-',
            alpha=0.9, 
            label='Avg Representation Std')
    
    ax.set_title(f"[{name}] Standard Deviation of O-Proj Input\n(Layer {layer})", 
                 fontsize=24, pad=20, fontweight='bold')
    ax.set_xlabel("Token Position", fontsize=22)
    ax.set_ylabel("Representation Std Dev", fontsize=22)
    
    ax.grid(True, linestyle='--', alpha=0.5)
    
    first_val = std_data[0]
    ax.annotate(f"{first_val:.2f}", xy=(0, first_val), 
                xytext=(5, first_val + (first_val * 0.05)),
                arrowprops=dict(facecolor='black', arrowstyle='->', lw=2.5),
                fontsize=20, fontweight='bold')

    ax.set_xlim(-2, SEQ_LEN + 2)
    
    ax.legend(loc='upper right', frameon=True, fontsize=16)
    
    print("[Plot] Showing averaged analysis figure...")
    plt.show()

if __name__ == "__main__":
    all_results = []
    
    for config in CONFIGS:
        std_data = get_avg_oproj_input_std(config, SEQ_LEN, NUM_SAMPLES, DEVICE)
        all_results.append((config, std_data))
        
    plot_std_analysis_single(all_results)

## Super Neuron Identification

Aligns with Sec. 4.2 and Fig. 8. Finds FFN gate/up neurons with unusually large weight norms.


In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from transformers import AutoModelForCausalLM
import gc
import os

# =========================================
# =========================================
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 26,               # 22 -> 26
    'axes.labelsize': 28,          # 24 -> 28
    'axes.titlesize': 30,          # 26 -> 30
    'xtick.labelsize': 24,         # 20 -> 24
    'ytick.labelsize': 24,         # 20 -> 24
    'figure.titlesize': 32,        # 28 -> 32
    'lines.linewidth': 3,
    'lines.markersize': 10,
    'grid.alpha': 0.4,
})

MODEL_PATH = os.getenv("LLAMA2_MODEL_PATH", "meta-llama/Llama-2-7b-hf")
TARGET_LAYER = 1  
DEVICE = os.getenv("DEVICE", "cuda:0" if torch.cuda.is_available() else "cpu") 

def analyze_mlp_weights(model_path, layer_idx, device):
    print(f"Loading model from {model_path}...")
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        device_map=device,
        torch_dtype=torch.float16,
        trust_remote_code=True
    )
    model.eval()

    mlp = model.model.layers[layer_idx].mlp
    
    print(f"Analyzing Layer {layer_idx} MLP...")
    
    w_up = mlp.up_proj.weight.detach().float() 
    norm_up = torch.norm(w_up, p=2, dim=1).cpu().numpy()
    
    w_gate = mlp.gate_proj.weight.detach().float()
    norm_gate = torch.norm(w_gate, p=2, dim=1).cpu().numpy()
    
    top_indices_up = np.argsort(norm_up)[::-1][:5]
    top_indices_gate = np.argsort(norm_gate)[::-1][:5]
    
    print("\n" + "="*40)
    print(f"TOP 5 Neurons in W_UP (Layer {layer_idx})")
    print("="*40)
    print(f"{'Rank':<5} | {'Neuron ID':<10} | {'L2 Norm':<10}")
    print("-" * 30)
    for i, idx in enumerate(top_indices_up):
        print(f"{i+1:<5} | {idx:<10} | {norm_up[idx]:.4f}")
        
    print("\n" + "="*40)
    print(f"TOP 5 Neurons in W_GATE (Layer {layer_idx})")
    print("="*40)
    print(f"{'Rank':<5} | {'Neuron ID':<10} | {'L2 Norm':<10}")
    print("-" * 30)
    for i, idx in enumerate(top_indices_gate):
        print(f"{i+1:<5} | {idx:<10} | {norm_gate[idx]:.4f}")

    del model
    gc.collect()
    torch.cuda.empty_cache()
    
    return norm_up, norm_gate, top_indices_up, top_indices_gate

def plot_weight_norms(norm_data, top_indices, name, color, layer_idx):
    """
    ， Top 5
    """
    sorted_indices = np.argsort(norm_data)[::-1]
    sorted_norms = norm_data[sorted_indices]
    
    fig, ax = plt.subplots(figsize=(12, 8), constrained_layout=True)
    
    x_axis = np.arange(len(sorted_norms))
    
    ax.plot(x_axis, sorted_norms, color=color, linewidth=3, label='Neuron Weight Norm')
    
    avg_norm = np.mean(sorted_norms)
    ax.axhline(avg_norm, color='black', linestyle='--', linewidth=2, label=f'Average: {avg_norm:.2f}')
    
    ax.plot(np.arange(5), sorted_norms[:5], 'o', color='red', markersize=12, 
            markeredgecolor='black', markeredgewidth=2, label='Top 5 Super Neurons')
    
    top1_id = sorted_indices[0]
    top1_val = sorted_norms[0]
    ax.annotate(f"ID: {top1_id}\nNorm: {top1_val:.2f}", 
                xy=(0, top1_val), 
                xytext=(500, top1_val), 
                arrowprops=dict(facecolor='black', arrowstyle='->', lw=2),
                fontsize=26, fontweight='bold')

    ax.set_title(f"[{name}] Neuron Weight Norm Distribution\n(Layer {layer_idx+1}, Sorted)", fontsize=29, pad=20, fontweight='bold')
    ax.set_xlabel("Neuron Rank (Sorted by Norm)", fontsize=30)
    ax.set_ylabel("L2 Norm of Row Vector", fontsize=30)
    ax.set_xlim(-100, len(sorted_norms)+100)
    
    ax.legend(fontsize=26, loc='upper right')
    ax.grid(True, linestyle='--', alpha=0.5)
    
    plt.show()

if __name__ == "__main__":
    norm_up, norm_gate, top_idx_up, top_idx_gate = analyze_mlp_weights(MODEL_PATH, TARGET_LAYER, DEVICE)
    
    plot_weight_norms(norm_up, top_idx_up, "W_UP (Up Projection)", "#1f77b4", TARGET_LAYER)
    plot_weight_norms(norm_gate, top_idx_gate, "W_GATE (Gate Projection)", "#ff7f0e", TARGET_LAYER)

## Selective Super Neuron Activation

Aligns with Sec. 4.2 and Fig. 9. Measures gate alignment and up-projection activation for a selected super neuron.


In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from tqdm import tqdm
import gc
import os

# ==========================================================
# ==========================================================
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 26,
    'axes.labelsize': 28,
    'axes.titlesize': 32,
    'xtick.labelsize': 24,
    'ytick.labelsize': 24,
    'legend.fontsize': 24,
    'axes.labelweight': 'bold',
    'axes.titleweight': 'bold',
    'lines.linewidth': 4,
    'lines.markersize': 12,
    'grid.alpha': 0.4,
})

MODEL_PATH = os.getenv("LLAMA2_MODEL_PATH", "meta-llama/Llama-2-7b-hf")
DEVICE = os.getenv("DEVICE", "cuda:0" if torch.cuda.is_available() else "cpu")
TARGET_LAYER = 1      
TARGET_NEURON_ID = 7890 
SEQ_LEN = 64
NUM_SAMPLES = 20      

# =========================================
# =========================================
def get_wikitext_samples(tokenizer, num_samples, seq_len):
    print(f"Loading WikiText-2 samples...")
    samples = []
    try:
        data = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
        texts = [x['text'] for x in data if len(x['text']) > seq_len * 4]
        indices = np.random.permutation(len(texts))
        for idx in indices:
            if len(samples) >= num_samples: break
            tokens = tokenizer(texts[idx], return_tensors="pt").input_ids
            if tokens.shape[1] >= seq_len:
                samples.append(tokens[:, :seq_len])
    except: pass
    if len(samples) < num_samples:
        while len(samples) < num_samples:
            samples.append(torch.randint(0, 32000, (1, seq_len)))
    return samples

def get_averaged_neuron_metrics(model_path, layer_idx, neuron_idx, seq_len, num_samples, device):
    print(f"Processing Model Metrics...")
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_path, device_map=device, torch_dtype=torch.float16, trust_remote_code=True
    ).eval()

    samples = get_wikitext_samples(tokenizer, num_samples, seq_len)
    mlp = model.model.layers[layer_idx].mlp
    w_gate_vec = mlp.gate_proj.weight[neuron_idx].detach()
    w_up_vec = mlp.up_proj.weight[neuron_idx].detach()

    current_state = {"input": None}
    def hook_fn(module, input, output):
        current_state["input"] = input[0].detach()

    handle = mlp.register_forward_hook(hook_fn)
    accumulated_cos, accumulated_act = np.zeros(seq_len), np.zeros(seq_len)
    
    with torch.no_grad():
        for inp in tqdm(samples):
            model(inp.to(device))
            x = current_state["input"][0] 
            cos_sim = F.cosine_similarity(x, w_gate_vec.unsqueeze(0), dim=-1).float().cpu().numpy()
            raw_act = torch.matmul(x, w_up_vec).float().cpu().numpy()
            accumulated_cos += cos_sim
            accumulated_act += raw_act

    handle.remove()
    res = (accumulated_cos/len(samples), accumulated_act/len(samples))
    del model, tokenizer
    torch.cuda.empty_cache()
    gc.collect()
    return res

# ==========================================================
# ==========================================================

def plot_synergy_analysis(cos_sim, activation, neuron_id, layer_idx):
    fig, ax1 = plt.subplots(figsize=(15, 9), constrained_layout=True)
    
    color_gate, color_up = '#1f77b4', '#d62728'
    x_axis = np.arange(len(cos_sim))

    line1 = ax1.plot(x_axis, cos_sim, color=color_gate, linestyle='-', 
                     marker='s', markersize=10, alpha=0.9, zorder=3,
                     label=f'Avg Cosine Sim (Gate)')
    
    ax1.set_xlabel('Token Position', fontweight='bold')
    ax1.set_ylabel('Cosine Similarity', color=color_gate, fontweight='bold')
    ax1.tick_params(axis='y', labelcolor=color_gate)
    ax1.grid(True, linestyle='--', alpha=0.4)

    tick_positions = np.arange(0, len(cos_sim), 10)
    if (len(cos_sim)-1) not in tick_positions:
        tick_positions = np.append(tick_positions, len(cos_sim)-1)
    
    ax1.set_xticks(tick_positions)
    ax1.set_xticklabels([str(p) for p in tick_positions])

    ax2 = ax1.twinx()
    line2 = ax2.plot(x_axis, activation, color=color_up, linestyle='--', 
                     marker='o', markersize=10, linewidth=4, alpha=0.9, zorder=3,
                     label=f'Avg Raw Activation (Up)')
    
    ax2.set_ylabel('Raw Activation Value', color=color_up, fontweight='bold')
    ax2.tick_params(axis='y', labelcolor=color_up)
    
    ax1.set_title(f"Synergistic Effect: Neuron {neuron_id} (Layer {layer_idx+1})", 
                  fontweight='bold', pad=30)
    
    val_act, val_cos = activation[0], cos_sim[0]
    
    ax2.annotate(f"{val_act:.1f}", xy=(0, val_act), xytext=(12, val_act), 
                  arrowprops=dict(facecolor=color_up, arrowstyle='->', lw=2.5),
                  color=color_up, fontsize=26, fontweight='bold')
    
    ax1.annotate(f"{val_cos:.2f}", xy=(0, val_cos), xytext=(12, val_cos - 0.05), 
                  arrowprops=dict(facecolor=color_gate, arrowstyle='->', lw=2.5),
                  color=color_gate, fontsize=26, fontweight='bold')

    lines = line1 + line2
    labels = [l.get_label() for l in lines]
    ax1.legend(lines, labels, loc='center right', fontsize=24, 
               frameon=True, fancybox=True, framealpha=1.0, edgecolor='black', shadow=False)
    
    ax1.set_xlim(-0.5, len(cos_sim) + 0.5) 
    
    plt.show()

if __name__ == "__main__":
    avg_cos, avg_act = get_averaged_neuron_metrics(MODEL_PATH, TARGET_LAYER, TARGET_NEURON_ID, SEQ_LEN, NUM_SAMPLES, DEVICE)
    plot_synergy_analysis(avg_cos, avg_act, TARGET_NEURON_ID, TARGET_LAYER)

## Sparse Down-Projection Channeling

Aligns with Sec. 4.2 and Fig. 10. Shows how a super neuron's activation is routed into a small set of output dimensions.


In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from transformers import AutoModelForCausalLM
import gc
from scipy.stats import kurtosis
import os

# =========================================
# =========================================
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 24,               # 18 -> 24
    'axes.labelsize': 26,          # 20 -> 26
    'axes.titlesize': 28,          # 22 -> 28
    'xtick.labelsize': 22,         # 16 -> 22
    'ytick.labelsize': 22,         # 16 -> 22
    'figure.titlesize': 30,        # 24 -> 30
    'lines.linewidth': 3,
    'lines.markersize': 10,
    'grid.alpha': 0.4,
})

MODEL_PATH = os.getenv("LLAMA2_MODEL_PATH", "meta-llama/Llama-2-7b-hf")
DEVICE = os.getenv("DEVICE", "cuda:0" if torch.cuda.is_available() else "cpu")
TARGET_LAYER = 1        # Llama Layer 1
TARGET_NEURON_ID = 7890

def analyze_down_proj_channeling(model_path, layer_idx, neuron_idx, device):
    print(f"Loading model from {model_path}...")
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        device_map=device,
        torch_dtype=torch.float16,
        trust_remote_code=True
    )
    model.eval()

    print(f"Analyzing Down-Projection for Neuron {neuron_idx} in Layer {layer_idx}...")
    
    mlp = model.model.layers[layer_idx].mlp
    w_down = mlp.down_proj.weight.detach().float()
    
    neuron_output_weights = w_down[:, neuron_idx].cpu().numpy() # Shape: (4096,)
    
    l2_norm = np.linalg.norm(neuron_output_weights)
    kurt_val = kurtosis(neuron_output_weights)
    
    top_k = 10
    sorted_indices = np.argsort(np.abs(neuron_output_weights))[::-1]
    top_dims = sorted_indices[:top_k]
    top_weights = neuron_output_weights[top_dims]
    
    print("\n" + "="*50)
    print(f"Down-Proj Analysis for Neuron {neuron_idx}")
    print("="*50)
    print(f"L2 Norm of Weight Vector: {l2_norm:.4f}")
    print(f"Kurtosis (Spikiness):     {kurt_val:.2f} (High value implies outliers)")
    print("-" * 50)
    print(f"{'Rank':<5} | {'Output Dim':<12} | {'Weight Value':<12}")
    print("-" * 50)
    for i in range(top_k):
        print(f"{i+1:<5} | {top_dims[i]:<12} | {top_weights[i]:.4f}")
        
    del model
    gc.collect()
    torch.cuda.empty_cache()
    
    return neuron_output_weights, top_dims, top_weights, kurt_val

def plot_channeling_effect(weights, top_dims, neuron_id, layer_idx, kurt_val):
    """
     Down-Proj ， Channeling Effect
    """
    sorted_abs_weights = np.sort(np.abs(weights))[::-1]
    
    fig, ax = plt.subplots(figsize=(14, 8), constrained_layout=True)
    
    x_axis = np.arange(len(weights))
    
    ax.plot(x_axis, sorted_abs_weights, color='#2ca02c', linewidth=3, label='Weight Magnitude (Sorted)')
    
    ax.fill_between(x_axis, sorted_abs_weights, color='#2ca02c', alpha=0.1)
    
    ax.plot(np.arange(5), sorted_abs_weights[:5], 'o', color='red', markersize=12, 
            markeredgecolor='black', markeredgewidth=2, label='Top 5 Outlier Weights')
    
    top1_val = sorted_abs_weights[0]
    ax.annotate(f"Max Weight: {top1_val:.4f}\n(Targeting Dim {top_dims[0]})", 
                xy=(0, top1_val), 
                xytext=(150, top1_val-0.25), 
                arrowprops=dict(facecolor='black', arrowstyle='->', lw=2),
                fontsize=24, fontweight='bold')
    
    textstr = '\n'.join((
        f'Neuron ID: {neuron_id}',
        f'Kurtosis: {kurt_val:.1f}',
        r'Distribution: Heavy-tailed'
    ))
    props = dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='gray')
    ax.text(0.5, 0.95, textstr, transform=ax.transAxes, fontsize=24,
            verticalalignment='top', bbox=props)

    ax.set_title(f"Channeling Effect: Down-Projection Weights of Neuron {neuron_id}\n(Layer {layer_idx})", 
                  fontweight='bold', pad=20)
    ax.set_xlabel("Output Dimension Rank (Sorted by Magnitude)")
    ax.set_ylabel("Absolute Weight Value")
    
    ax.set_xlim(-10, 1000) 
    
    ax.legend(fontsize=24, loc='center right')
    ax.grid(True, linestyle='--', alpha=0.5)
    
    print(f"[Plot] Showing Down-Proj channeling analysis...")
    plt.show()

if __name__ == "__main__":
    weights, top_dims, top_vals, kurt = analyze_down_proj_channeling(MODEL_PATH, TARGET_LAYER, TARGET_NEURON_ID, DEVICE)
    
    plot_channeling_effect(weights, top_dims, TARGET_NEURON_ID, TARGET_LAYER, kurt)

## Dimension Disparity After RMSNorm

Aligns with Sec. 4.3 and Table 1. Quantifies dominance of a selected outlier dimension after RMSNorm.


In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import gc
import os

# =========================================
# =========================================
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 18,
    'axes.labelsize': 20,
    'axes.titlesize': 22,
    'xtick.labelsize': 18,
    'ytick.labelsize': 18,
    'figure.titlesize': 24,
    'grid.alpha': 0.4,
})

MODEL_PATH = os.getenv("LLAMA2_MODEL_PATH", "meta-llama/Llama-2-7b-hf")
DEVICE = os.getenv("DEVICE", "cuda:0" if torch.cuda.is_available() else "cpu")
TARGET_LAYER = 2         # Layer 2
TARGET_DIM = 2533
SEQ_LEN = 64
NUM_SAMPLES = 20

# =========================================
# =========================================
def get_wikitext_samples(tokenizer, num_samples, seq_len):
    print(f"Loading WikiText-2 samples...")
    samples = []
    try:
        data = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
        texts = [x['text'] for x in data if len(x['text']) > seq_len * 4]
        indices = np.random.permutation(len(texts))
        for idx in indices:
            if len(samples) >= num_samples: break
            tokens = tokenizer(texts[idx], return_tensors="pt").input_ids
            if tokens.shape[1] >= seq_len:
                samples.append(tokens[:, :seq_len])
    except Exception as e:
        print(f"Error: {e}")
        
    while len(samples) < num_samples:
        vocab_size = getattr(tokenizer, "vocab_size", 32000)
        samples.append(torch.randint(0, vocab_size, (1, seq_len)))
    return samples

# =========================================
# =========================================
def analyze_specific_dim_activation(model_path, layer_idx, target_dim, device):
    print(f"Loading model from {model_path}...")
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        device_map=device,
        torch_dtype=torch.float16,
        trust_remote_code=True
    )
    model.eval()

    samples = get_wikitext_samples(tokenizer, NUM_SAMPLES, SEQ_LEN)
    
    target_module = model.model.layers[layer_idx].input_layernorm
    
    stats = {
        "target_abs": [],
        "others_abs_mean": []
    }
    
    def hook_fn(module, input, output):
        # output Shape: [Batch, Seq, Hidden] -> [1, 64, 4096]
        token0_feat = output[0, 0, :] # [4096]
        
        val_target = token0_feat[target_dim].abs().item()
        
        sum_abs_all = token0_feat.abs().sum().item()
        mean_others = (sum_abs_all - val_target) / (token0_feat.shape[0] - 1)
        
        stats["target_abs"].append(val_target)
        stats["others_abs_mean"].append(mean_others)

    handle = target_module.register_forward_hook(hook_fn)
    
    print(f"Running inference on {NUM_SAMPLES} samples...")
    with torch.no_grad():
        for inp in tqdm(samples):
            inp = inp.to(device)
            model(inp)
            
    handle.remove()
    
    avg_target_abs = np.mean(stats["target_abs"])
    avg_others_abs = np.mean(stats["others_abs_mean"])
    
    ratio = avg_target_abs / avg_others_abs
    
    print("\n" + "="*40)
    print(f"Layer {layer_idx} Pre-Attention Output Analysis (Token 0)")
    print("="*40)
    print(f"Target Dim ({target_dim}) Abs Value : {avg_target_abs:.4f}")
    print(f"Other Dims Avg Abs Value        : {avg_others_abs:.4f}")
    print(f"Ratio (Target / Others)         : {ratio:.2f}x")
    print("="*40)
    
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    
    return avg_target_abs, avg_others_abs, ratio

# =========================================
# =========================================
def plot_comparison_bar(val_target, val_others, layer_idx, target_dim):
    fig, ax = plt.subplots(figsize=(10, 8), constrained_layout=True)
    
    categories = [f'Target Dim\n({target_dim})', 'Avg of\nOther Dims']
    values = [val_target, val_others]
    colors = ['#d62728', '#7f7f7f']
    
    bars = ax.bar(categories, values, color=colors, alpha=0.9, width=0.6, edgecolor='black', linewidth=2)
    
    ax.set_ylabel("Absolute Activation Value", fontsize=22)
    ax.set_title(f"[Llama-2-7b] Token 0 Representation Magnitude\n(Layer {layer_idx} Pre-Attention Output)", fontsize=24, pad=20, fontweight='bold')
    
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.2f}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 5),
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=20, fontweight='bold')
        
    ratio = val_target / val_others
    if ratio > 1:
        mid_x = (bars[0].get_x() + bars[1].get_x() + bars[0].get_width()) / 2 + 0.1
        mid_y = (val_target + val_others) / 2
        
        ax.annotate(f"{ratio:.1f}x Larger", 
                    xy=(0, val_others), xytext=(0.5, val_target),
                    arrowprops=dict(arrowstyle='|-|', color='black', lw=2),
                    fontsize=20, ha='center', va='center', fontweight='bold', color='black')

    ax.grid(True, axis='y', linestyle='--', alpha=0.5)
    
    print(f"[Plot] Showing bar comparison...")
    plt.show()

if __name__ == "__main__":
    val_t, val_o, r = analyze_specific_dim_activation(MODEL_PATH, TARGET_LAYER, TARGET_DIM, DEVICE)
    
    # plot_comparison_bar(val_t, val_o, TARGET_LAYER, TARGET_DIM)

## QK Structural Locking

Aligns with Sec. 4.3 and Fig. 12. Compares sink-key alignment with query principal directions and positive attention-score ratios.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import gc
from tqdm import tqdm
import os

# =========================================
# =========================================
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 18,
    'axes.labelsize': 20,
    'axes.titlesize': 22,
    'xtick.labelsize': 16,
    'ytick.labelsize': 16,
    'figure.titlesize': 24,
    'lines.linewidth': 4, 
    'lines.markersize': 12,
    'grid.alpha': 0.4,
    'mathtext.fontset': 'stix', 
})

model_path = os.getenv("LLAMA2_MODEL_PATH", "meta-llama/Llama-2-7b-hf")
TARGET_LAYER_IDX = 2
OUTLIER_DIM_IDX = 2533 
SEQ_LEN = 64          
NUM_SAMPLES = 20
DEVICE = os.getenv("DEVICE", "cuda:0" if torch.cuda.is_available() else "cpu") 

# =========================================
# =========================================
def analyze_per_head_mechanics(model_path, layer_idx, dim_idx, device):
    print(f"Loading model from {model_path}...")
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForCausalLM.from_pretrained(
        model_path, 
        device_map=device, 
        torch_dtype=torch.float16
    )
    model.eval()
    
    config = model.config
    num_heads = config.num_attention_heads      # 32
    head_dim = config.hidden_size // num_heads  # 128

    print("Sampling WikiText-2...")
    dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
    long_text = [x['text'] for x in dataset if len(x['text']) > SEQ_LEN * 4]
    indices = np.random.choice(len(long_text), NUM_SAMPLES, replace=False)
    
    samples_tokens = []
    for idx in indices:
        txt = long_text[idx]
        tokens = tokenizer(txt, return_tensors="pt").input_ids
        if tokens.shape[1] >= SEQ_LEN:
            samples_tokens.append(tokens[:, :SEQ_LEN])

    layer = model.model.layers[layer_idx]
    W_Q_global = layer.self_attn.q_proj.weight.detach().float().cpu().numpy()
    W_K_global = layer.self_attn.k_proj.weight.detach().float().cpu().numpy()

    k_response_vector = W_K_global[:, dim_idx] 
    k_heads = k_response_vector.reshape(num_heads, head_dim) 
    
    k_heads_norm = np.zeros_like(k_heads)
    for h in range(num_heads):
        norm = np.linalg.norm(k_heads[h])
        if norm > 1e-9:
            k_heads_norm[h] = k_heads[h] / norm

    print("Computing SVD per head...")
    rank1_alignments_abs = np.zeros(num_heads)

    for h in range(num_heads):
        start_row = h * head_dim
        end_row = (h + 1) * head_dim
        W_Q_head = W_Q_global[start_row:end_row, :]
        
        U, S, Vt = np.linalg.svd(W_Q_head, full_matrices=False)
        u_1st = U[:, 0]
        
        if np.linalg.norm(k_heads_norm[h]) > 1e-9:
            rank1_alignments_abs[h] = np.abs(np.dot(u_1st, k_heads_norm[h]))
        else:
            rank1_alignments_abs[h] = 0.0

    print("Collecting runtime QK dot products...")
    captured_q_heads = [] 
    
    def hook_q(module, input, output):
        q_act = output.detach().cpu().float()[:, 1:, :]
        batch, seq_len, _ = q_act.shape
        captured_q_heads.append(q_act.view(batch, seq_len, num_heads, head_dim).reshape(-1, num_heads, head_dim))

    handle = layer.self_attn.q_proj.register_forward_hook(hook_q)
    with torch.no_grad():
        for tokens in tqdm(samples_tokens):
            inputs = tokens.to(device)
            model(inputs)
    handle.remove()

    q_vectors_mha = torch.cat(captured_q_heads, dim=0).numpy()
    
    dot_products_per_head = np.einsum('nhd,hd->nh', q_vectors_mha, k_heads)
    
    total_count = dot_products_per_head.shape[0]
    positive_counts = np.sum(dot_products_per_head > 0, axis=0)
    positive_ratios = positive_counts / total_count * 100.0 

    del model, tokenizer
    torch.cuda.empty_cache()
    gc.collect()

    return rank1_alignments_abs, positive_ratios, dim_idx

# =========================================
# =========================================

def plot_dual_axis_analysis(rank1_aligns_abs, positive_ratios, dim_idx):
    fig, ax1 = plt.subplots(figsize=(14, 8), constrained_layout=True)
    
    head_indices = np.arange(len(rank1_aligns_abs))
    
    # ==========================
    # ==========================
    color_bar = '#1f77b4'
    bars = ax1.bar(head_indices, rank1_aligns_abs, color=color_bar, alpha=0.5, 
                   edgecolor='black', linewidth=1.2, label='Structural Potential (Alignment)')
    
    ax1.set_xlabel("Head Index (0-31)", fontsize=22)
    ax1.set_ylabel(r"Rank-1 Alignment $|\langle \mathbf{u}_1, \mathbf{k} \rangle|$", 
                   fontsize=22, color='darkgreen')
    ax1.tick_params(axis='y', labelcolor='darkgreen')
    
    ax1.set_ylim(0, 0.6)
    ax1.set_yticks(np.arange(0, 0.61, 0.1))
    
    ax1.grid(axis='y', linestyle='--', alpha=0.4) 

    # ==========================
    # ==========================
    ax2 = ax1.twinx()  
    
    color_line = '#d62728'
    line = ax2.plot(head_indices, positive_ratios, color=color_line, marker='o', 
                    linestyle='-', label='Actual Activation (Positive %)', zorder=10)
    
    ax2.set_ylabel(r"Positive Dot Product Ratio (%)", fontsize=22, color='darkblue')
    ax2.tick_params(axis='y', labelcolor='darkblue')
    
    ax2.set_ylim(80, 100)
    ax2.set_yticks(np.arange(80, 101, 5))

    # ==========================
    # ==========================
    
    ax1.set_xticks(head_indices)
    ax1.set_xticklabels([str(i) for i in head_indices], fontsize=14, rotation=0)

    lines_1, labels_1 = ax1.get_legend_handles_labels()
    lines_2, labels_2 = ax2.get_legend_handles_labels()
    
    ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper left', fontsize=16, framealpha=0.9)

    plt.title(f"Mechanism Verification: Structure vs. Activation (Input Dim {dim_idx})", 
              fontsize=24, fontweight='bold', pad=20)

    print("[Plot] Showing Dual-Axis Analysis (Zoomed)...")
    plt.show()

if __name__ == "__main__":
    aligns_abs, pos_ratios, dim = analyze_per_head_mechanics(model_path, TARGET_LAYER_IDX, OUTLIER_DIM_IDX, DEVICE)
    plot_dual_axis_analysis(aligns_abs, pos_ratios, dim)

## Toy Model: Aggregation Variance Decay

Supports Sec. 3.1. Uses a controlled Llama-style block to compare empirical value-aggregation variance with the theoretical decay curve.


In [ ]:
import math
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

# =========================================
# =========================================
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 18,
    'axes.labelsize': 20,
    'axes.titlesize': 22,
    'xtick.labelsize': 16,
    'ytick.labelsize': 16,
    'figure.titlesize': 24,
    'lines.linewidth': 3,
    'lines.markersize': 10,
    'grid.alpha': 0.4,
    'mathtext.fontset': 'stix',
})

DEVICE = "cpu"
torch.manual_seed(42)
np.random.seed(42)

# -----------------------
# RoPE (cache + apply)
# -----------------------
class RoPE:
    def __init__(self, head_dim: int, max_seq_len: int = 2048, base: int = 10000):
        assert head_dim % 2 == 0
        theta = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        seq_idx = torch.arange(max_seq_len).float()
        idx_theta = torch.outer(seq_idx, theta).float()
        cache = torch.stack([torch.cos(idx_theta), torch.sin(idx_theta)], dim=-1)
        self.cache = cache

    def apply(self, x: torch.Tensor):
        T = x.size(1)
        rope = self.cache[:T].to(x.device)
        x_shaped = x.reshape(*x.shape[:-1], -1, 2)
        rope_view = rope.view(1, rope.size(0), 1, rope.size(1), 2)
        x_out = torch.stack(
            [
                x_shaped[..., 0] * rope_view[..., 0] - x_shaped[..., 1] * rope_view[..., 1],
                x_shaped[..., 1] * rope_view[..., 0] + x_shaped[..., 0] * rope_view[..., 1],
            ],
            dim=-1,
        )
        x_out = x_out.flatten(-2)
        return x_out.type_as(x)

# -----------------------
# RMSNorm
# -----------------------
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x: torch.Tensor):
        norm_x = torch.mean(x * x, dim=-1, keepdim=True)
        x_normed = x * torch.rsqrt(norm_x + self.eps)
        return self.scale * x_normed

# -----------------------
# Attention with causal mask
# -----------------------
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd: int, n_head: int, max_seq_len: int):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.n_embd = n_embd
        self.head_dim = n_embd // n_head
        
        # Linear layers
        self.c_attn = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.c_proj = nn.Linear(n_embd, n_embd, bias=False)
        
        self.rope = RoPE(self.head_dim, max_seq_len=max_seq_len)
        self.max_seq_len = max_seq_len

        init_std = 0.02 / np.sqrt(2 * 12)
        nn.init.normal_(self.c_attn.weight, mean=0.0, std=init_std)
        nn.init.normal_(self.c_proj.weight, mean=0.0, std=init_std)

    def forward(self, x: torch.Tensor):
        B, T, C = x.size()
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)

        q = q.view(B, T, self.n_head, self.head_dim)
        k = k.view(B, T, self.n_head, self.head_dim)
        v = v.view(B, T, self.n_head, self.head_dim)

        q = self.rope.apply(q)
        k = self.rope.apply(k)

        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        
        # Causal Mask
        causal_mask = torch.tril(torch.ones(T, T, device=x.device, dtype=torch.bool)).view(1, 1, T, T)
        att = att.masked_fill(~causal_mask, -1e9)
        att = F.softmax(att, dim=-1)

        # Value Aggregation
        y = att @ v  # (B, H, T, head_dim)
        y = y.transpose(1, 2).contiguous().view(B, T, C)

        # Capture pre-projection y (this is the aggregated value V_out)
        y_pre_proj = y.clone()
        y = self.c_proj(y)

        return y, att, y_pre_proj

# -----------------------
# LLaMA-style Block
# -----------------------
class Block(nn.Module):
    def __init__(self, n_embd: int, n_head: int, max_seq_len: int):
        super().__init__()
        self.rms_1 = RMSNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, max_seq_len)
        self.rms_2 = RMSNorm(n_embd)
        self.mlp = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd, bias=False),
            nn.SiLU(),
            nn.Linear(4*n_embd, n_embd, bias=False),
        )
        # Init MLP weights
        for m in self.mlp.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, mean=0.0, std=0.02 / np.sqrt(2 * 12))

    def forward(self, x: torch.Tensor):
        h, att, y_pre = self.attn(self.rms_1(x))
        x = x + h
        x = x + self.mlp(self.rms_2(x))
        return x, att, y_pre

# -----------------------
# Experiment runner
# -----------------------
def run_std_vs_position(n_embd=768, n_head=12, seq_len=64, n_layers=1, device="cpu"):
    B = 1
    C = n_embd
    
    # === 1. Input Initialization ===
    x = torch.randn(B, seq_len, C, device=device) * 0.02 
    
    blocks = nn.ModuleList([Block(n_embd, n_head, seq_len) for _ in range(n_layers)]).to(device)

    # === 2. Forward Pass ===
    y_pre_by_layer = []
    with torch.no_grad():
        for i, block in enumerate(blocks):
            x_out, att, y_pre = block(x)
            y_pre_by_layer.append(y_pre.squeeze(0).cpu().numpy())
            x = x_out

    # === 3. Analysis (Actual) ===
    y_pre_last = y_pre_by_layer[-1]  # (T, C)
    actual_stds = np.std(y_pre_last, axis=1)

    # === 4. Theoretical Curve ===
    # theoretical (example): v_std_el = sqrt(C * input_std^2 * linear_weight_std^2)
    input_std = 1.0
    linear_weight_std = 0.02 / np.sqrt(2 * 12)
    
    real_input_std = 1
    
    v_var_el = C * (real_input_std ** 2) * (linear_weight_std ** 2)
    v_std_el = math.sqrt(v_var_el)
    
    # Aggregation Decay: sigma / sqrt(t)
    theoretical = np.array([v_std_el / math.sqrt(i + 1) for i in range(seq_len)])

    return actual_stds, theoretical

# -----------------------
# Plotting
# -----------------------
if __name__ == "__main__":
    seq_len = 64
    actual, theory = run_std_vs_position(
        n_embd=768, n_head=12, seq_len=seq_len, n_layers=1, device=DEVICE
    )

    fig, ax = plt.subplots(figsize=(14, 8), constrained_layout=True)

    # Plot Actual Data
    ax.plot(np.arange(1, seq_len+1), actual, 
            color='#1f77b4', marker='o', markersize=8, 
            label="Actual Std (Simulation)", alpha=0.9)

    # Plot Theoretical Curve
    ax.plot(np.arange(1, seq_len+1), theory, 
            color='#d62728', linestyle='--', linewidth=3, 
            label=r"Theory: $\sigma_v / \sqrt{t}$")

    # Annotations
    ax.set_title("Standard Deviation of Attention Output vs. Token Position", fontweight='bold', pad=20)
    ax.set_xlabel("Token Position $t$", fontsize=20)
    ax.set_ylabel("Representation Std Dev", fontsize=20)
    
    # Explicit Annotation for Init Std
    ax.text(0.55, 0.6, 
            f"Configuration:\n"
            # f"• Input Init $\sigma=0.02$\n"
            f"• Weight Init $\sigma=0.02/\sqrt{{2*L}}$\n"
            f"• $d_{{model}}=768$", 
            transform=ax.transAxes, fontsize=18,
            bbox=dict(boxstyle="round,pad=0.5", fc="white", ec="gray", alpha=0.9))

    # Arrow highlighting the first token
    first_val = actual[0]
    ax.annotate(f"Init Scale: {first_val:.4f}", 
                xy=(1, first_val), 
                xytext=(5, first_val + 0.002),
                arrowprops=dict(facecolor='black', arrowstyle='->', lw=2),
                fontsize=16, fontweight='bold')

    ax.legend(loc='upper right', fontsize=18, frameon=True)
    ax.grid(True, linestyle="--", alpha=0.5)
    
    print("[Plot] Showing verification figure...")
    plt.show()

## Effective Rank Dynamics

Aligns with Sec. 5.2 and Appendix C / Fig. 21. Computes layer-wise effective rank to diagnose manifold collapse.


In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import gc
import sys
import os

# =========================================
# =========================================
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 18,
    'axes.labelsize': 20,
    'axes.titlesize': 22,
    'xtick.labelsize': 16,
    'ytick.labelsize': 16,
    'figure.titlesize': 24,
    'lines.linewidth': 3,
    'lines.markersize': 9,
    'grid.alpha': 0.4,
    'mathtext.fontset': 'stix',
})

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_SAMPLES = 20
SEQ_LEN = 128

MODELS_CONFIG = [
    {
        "name": "Llama-2-7b",
        "path": os.getenv("LLAMA2_MODEL_PATH", "meta-llama/Llama-2-7b-hf"),
        "color": "#1f77b4",
        "marker": "o"
    },
    {
        "name": "Llama-3-8b",
        "path": os.getenv("LLAMA3_MODEL_PATH", "meta-llama/Meta-Llama-3-8B"),
        "color": "#ff7f0e",
        "marker": "s"
    }
]

# =========================================
# =========================================
def get_wikitext_samples(tokenizer, num_samples, max_len):
    print("Loading WikiText-2...")
    try:
        data = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
        min_char_len = max_len * 3.5 
        texts = [x['text'] for x in data if len(x['text']) > min_char_len]
        
        samples = []
        indices = np.random.permutation(len(texts))
        for idx in indices:
            if len(samples) >= num_samples: break
            tokens = tokenizer(texts[idx], return_tensors="pt").input_ids
            if tokens.shape[1] >= max_len:
                samples.append(tokens[:, :max_len])
        
        while len(samples) < num_samples:
            samples.append(torch.randint(0, 32000, (1, max_len)))
        return samples
    except Exception as e:
        print(f"Data error: {e}. Using dummy data.")
        return [torch.randint(0, 32000, (1, max_len)) for _ in range(num_samples)]

# =========================================
# =========================================

def compute_effective_rank(tensor):
    """  """
    # tensor: [Batch, Seq, Hidden] -> take Batch 0 -> [Seq, Hidden]
    if tensor.dim() == 3:
        matrix = tensor[0].float()
    else:
        matrix = tensor.float()
        
    try:
        # singular values
        s_vals = torch.linalg.svdvals(matrix)
        # normalize
        s_sum = torch.sum(s_vals)
        if s_sum > 0:
            p = s_vals / s_sum
            # Shannon entropy
            entropy = -torch.sum(p * torch.log(p + 1e-12))
            # Exp(Entropy)
            return torch.exp(entropy).item()
        return 0.0
    except:
        return 0.0

def analyze_hf_model_stats(model_path, seq_len, num_samples, device):
    print(f"Loading model from {model_path}...")
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        device_map=device,
        torch_dtype=torch.float16,
        trust_remote_code=True
    )
    model.eval()
    
    samples = get_wikitext_samples(tokenizer, num_samples, seq_len)
    
    num_layers = model.config.num_hidden_layers
    
    layer_ranks = {i: [] for i in range(num_layers)}
    
    hooks = []
    
    def get_hook(idx):
        def hook(module, input, output):
            # HF output: (hidden_states, past_key_values, ...)
            if isinstance(output, tuple):
                h = output[0]
            else:
                h = output
            
            h = h.detach()
            rank = compute_effective_rank(h)
            layer_ranks[idx].append(rank)
        return hook

    if hasattr(model, "model") and hasattr(model.model, "layers"):
        layers_obj = model.model.layers
    elif hasattr(model, "layers"):
        layers_obj = model.layers
    else:
        raise ValueError("Unknown model structure")

    for i, layer in enumerate(layers_obj):
        hooks.append(layer.register_forward_hook(get_hook(i)))
        
    print(f"Scanning layers...")
    with torch.no_grad():
        for inp in tqdm(samples):
            model(inp.to(device))
            
    for h in hooks: h.remove()
    
    avg_ranks = [np.mean(layer_ranks[i]) for i in range(num_layers)]
    
    del model, tokenizer
    torch.cuda.empty_cache()
    gc.collect()
    
    return avg_ranks

# =========================================
# =========================================

def plot_effective_rank_comparison(results_dict, seq_len):
    """
    results_dict: { 'ModelName': [rank_layer_0, rank_layer_1, ...] }
    """
    fig, ax = plt.subplots(figsize=(12, 8), constrained_layout=True)
    
    for cfg in MODELS_CONFIG:
        name = cfg['name']
        if name not in results_dict: continue
        
        ranks = results_dict[name]
        layers = np.arange(len(ranks))
        
        ax.plot(layers, ranks, 
                label=name,
                color=cfg['color'], 
                marker=cfg['marker'],
                linestyle='-', 
                linewidth=3, 
                alpha=0.9)

    ax.axhline(y=seq_len, color='gray', linestyle='--', linewidth=2, label=f'Theoretical Max Rank ({seq_len})')
    
    ax.set_title("Layer-wise Effective Rank Comparison", fontweight='bold', pad=20)
    ax.set_xlabel("Layer Index", fontsize=22)
    ax.set_ylabel("Effective Rank", fontsize=22)
    
    ax.set_ylim(0, seq_len * 1.1)
    
    ax.grid(True, ls="--", alpha=0.5)
    ax.legend(loc='best', fontsize=18, frameon=True)
    
    print("[Plot] Showing Effective Rank comparison...")
    plt.show()

# =========================================
# =========================================
if __name__ == "__main__":
    all_results = {}
    
    for cfg in MODELS_CONFIG:
        print(f"\n=== Processing {cfg['name']} ===")
        ranks = analyze_hf_model_stats(cfg['path'], SEQ_LEN, NUM_SAMPLES, DEVICE)
        all_results[cfg['name']] = ranks
        
    plot_effective_rank_comparison(all_results, SEQ_LEN)

## Layer-Wise Sink Onset and Norm Spike

Aligns with Sec. 3 and Fig. 3 / Appendix Fig. 18. Tracks attention received by the sink token and the corresponding representation norm across layers.


In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import gc
import sys
import os

# =========================================
# =========================================
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 18,
    'axes.labelsize': 20,
    'axes.titlesize': 22,
    'xtick.labelsize': 16,
    'ytick.labelsize': 16,
    'legend.fontsize': 15,
    'figure.titlesize': 24,
    'lines.linewidth': 3,
    'lines.markersize': 9,
    'grid.alpha': 0.4,
    'mathtext.fontset': 'stix',
})

DEVICE = os.getenv("DEVICE", "cuda:0" if torch.cuda.is_available() else "cpu") 
NUM_SAMPLES = 20
SEQ_LEN = 64     

MODELS_CONFIG = [
    {
        "name": "Llama-2-7b",
        "path": os.getenv("LLAMA2_MODEL_PATH", "meta-llama/Llama-2-7b-hf")
    },
    {
        "name": "Llama-3-8b",
        "path": os.getenv("LLAMA3_MODEL_PATH", "meta-llama/Meta-Llama-3-8B")
    }
]

# =========================================
# =========================================
def get_wikitext_samples(tokenizer, num_samples, seq_len):
    print(f"Loading WikiText-2 samples...")
    samples = []
    try:
        data = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
        texts = [x['text'] for x in data if len(x['text']) > seq_len * 4]
        indices = np.random.permutation(len(texts))
        for idx in indices:
            if len(samples) >= num_samples: break
            tokens = tokenizer(texts[idx], return_tensors="pt").input_ids
            if tokens.shape[1] >= seq_len:
                samples.append(tokens[:, :seq_len])
    except Exception as e:
        print(f"Error: {e}")
        
    while len(samples) < num_samples:
        vocab_size = getattr(tokenizer, "vocab_size", 32000)
        samples.append(torch.randint(0, vocab_size, (1, seq_len)))
    return samples

# =========================================
# =========================================
class LayerwiseAnalyzer:
    def __init__(self, cfg):
        self.name = cfg['name']
        self.path = cfg['path']
        print(f"Loading {self.name}...")
        
        self.tokenizer = AutoTokenizer.from_pretrained(self.path, trust_remote_code=True)
        self.model = AutoModelForCausalLM.from_pretrained(
            self.path,
            device_map=DEVICE,
            torch_dtype=torch.float16,
            trust_remote_code=True,
            attn_implementation="eager" 
        )
        self.model.eval()
        self.num_layers = self.model.config.num_hidden_layers

    def run_analysis(self):
        samples = get_wikitext_samples(self.tokenizer, NUM_SAMPLES, SEQ_LEN)
        
        sink_scores = np.zeros(self.num_layers)
        base_scores = np.zeros(self.num_layers)
        sink_norms = np.zeros(self.num_layers)
        base_norms = np.zeros(self.num_layers)
        
        count = 0
        
        # Lower Triangular = 1 (Valid), Upper = 0 (Invalid)
        # Shape: [Seq, Seq]
        causal_mask = torch.tril(torch.ones((SEQ_LEN, SEQ_LEN), device=DEVICE, dtype=torch.bool))
        
        valid_others_mask = causal_mask.clone()
        valid_others_mask[:, 0] = False 
        
        print(f"Running analysis for {self.name}...")
        with torch.no_grad():
            for inp in tqdm(samples):
                inp = inp.to(DEVICE)
                outputs = self.model(inp, output_attentions=True, output_hidden_states=True)
                
                # --- 1. Attention Analysis ---
                for i, layer_attn in enumerate(outputs.attentions):
                    # layer_attn: [Batch=1, Heads, Seq, Seq] -> [Heads, Seq, Seq]
                    attn_matrix = layer_attn[0] 
                    
                    # --- A. Sink Score (Token 0) ---
                    sink_attn = attn_matrix[:, :, 0].mean().item()
                    sink_scores[i] += sink_attn
                    
                    valid_values = attn_matrix[:, valid_others_mask] 
                    base_attn = valid_values.mean().item()
                    base_scores[i] += base_attn
                
                # --- 2. Feature Norm Analysis ---
                for i in range(self.num_layers):
                    h_in = outputs.hidden_states[i] # [Batch, Seq, Hidden]
                    
                    # Sink Norm
                    norm_0 = torch.norm(h_in[0, 0, :], p=2).item()
                    sink_norms[i] += norm_0
                    
                    norm_others = torch.norm(h_in[0, 1:, :], p=2, dim=-1).mean().item()
                    base_norms[i] += norm_others
                    
                count += 1
        
        return (sink_scores/count, base_scores/count, 
                sink_norms/count, base_norms/count)

# =========================================
# =========================================
def plot_dual_axis_analysis(model_name, sink_scores, base_scores, sink_norms, base_norms):
    fig, ax1 = plt.subplots(figsize=(13, 8), constrained_layout=True)
    layers = np.arange(len(sink_scores))
    
    COLOR_SINK_ATTN = '#1f77b4' 
    COLOR_BASE_ATTN = '#aec7e8' 
    COLOR_SINK_NORM = '#d62728' 
    COLOR_BASE_NORM = '#ff9896' 
    
    # Left Axis: Attention
    l1 = ax1.plot(layers, sink_scores, color=COLOR_SINK_ATTN, linestyle='-', marker='o', 
                  linewidth=3, label='Attention to Token 0 (Sink)', zorder=10)
    l2 = ax1.plot(layers, base_scores, color='gray', linestyle=':', marker='.', 
                  linewidth=2, label='Avg Attention to Others', alpha=0.7)
    
    ax1.set_xlabel('Layer Depth', fontsize=22)
    ax1.set_ylabel('Attention Score', color=COLOR_SINK_ATTN, fontsize=22, fontweight='bold')
    ax1.tick_params(axis='y', labelcolor=COLOR_SINK_ATTN)
    ax1.set_ylim(-0.05, 1.05)
    ax1.grid(True, ls="--", alpha=0.3)

    # Right Axis: Norm
    ax2 = ax1.twinx()
    l3 = ax2.plot(layers, sink_norms, color=COLOR_SINK_NORM, linestyle='--', marker='s', 
                  linewidth=3, label='Token 0 Norm ($L_2$)', zorder=10)
    l4 = ax2.plot(layers, base_norms, color='gray', linestyle='--', marker='', 
                  linewidth=2, label='Avg Norm of Others', alpha=0.5)
    
    ax2.set_ylabel('Feature Norm Value', color=COLOR_SINK_NORM, fontsize=22, fontweight='bold')
    ax2.tick_params(axis='y', labelcolor=COLOR_SINK_NORM)
    
    # Annotations
    onset_candidates = np.where(sink_scores > 0.8)[0]
    if len(onset_candidates) > 0:
        onset_idx = onset_candidates[0]
        
        ax1.annotate(f"Sink Onset\n(Layer {onset_idx})", 
                     xy=(onset_idx, sink_scores[onset_idx]), 
                     xytext=(onset_idx + 3, 0.4),
                     arrowprops=dict(facecolor=COLOR_SINK_ATTN, arrowstyle='->', lw=2.5),
                     color=COLOR_SINK_ATTN, fontsize=16, fontweight='bold')
        
        norm_val = sink_norms[onset_idx]
        ax2.annotate(f"Massive Norm\n(Layer {onset_idx})", 
                     xy=(onset_idx, norm_val), 
                     xytext=(onset_idx + 3, norm_val),
                     arrowprops=dict(facecolor=COLOR_SINK_NORM, arrowstyle='->', lw=2.5),
                     color=COLOR_SINK_NORM, fontsize=16, fontweight='bold')
        
        ax1.axvline(x=onset_idx, color='black', linestyle='-.', alpha=0.3)

    ax1.set_title(f"[{model_name}] Outlier Analysis: Norms & Attention", 
                  fontsize=24, fontweight='bold', pad=20)
    
    lines = l1 + l2 + l3 + l4
    labels = [l.get_label() for l in lines]
    ax1.legend(lines, labels, loc='center right', fontsize=15, 
               frameon=True, facecolor='white', framealpha=0.9, edgecolor='gray')
    

    plt.show()

# =========================================
# =========================================
if __name__ == "__main__":
    
    for cfg in MODELS_CONFIG:
        analyzer = LayerwiseAnalyzer(cfg)
        s_scores, b_scores, s_norms, b_norms = analyzer.run_analysis()
        
        plot_dual_axis_analysis(cfg['name'], s_scores, b_scores, s_norms, b_norms)
        
        del analyzer
        gc.collect()
        torch.cuda.empty_cache()

## Random-Input Variance Validation

Aligns with Sec. 3.1 and Fig. 4. Uses random input tokens to remove semantic and BOS-token confounds when measuring dimension-wise variance.


In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm
import gc
import os

# =========================================
# =========================================
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 18,
    'axes.labelsize': 20,
    'axes.titlesize': 22,
    'xtick.labelsize': 16,
    'ytick.labelsize': 16,
    'figure.titlesize': 24,
    'lines.linewidth': 3,
    'lines.markersize': 10,
    'grid.alpha': 0.4,
})

CONFIG = {
    "name": "Llama-2-7b",
    "path": os.getenv("LLAMA2_MODEL_PATH", "meta-llama/Llama-2-7b-hf"),
    "target_layer": 1,
    "color": "#1f77b4"
}

SEQ_LEN = 64
NUM_SAMPLES = 500
DEVICE = os.getenv("DEVICE", "cuda:0" if torch.cuda.is_available() else "cpu") 

# =========================================
# =========================================

def get_random_input_dim_std(config, num_samples, seq_len, device):
    model_path = config["path"]
    layer_idx = config["target_layer"]
    
    print(f"Loading {config['name']} from {model_path}...")
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        device_map=device,
        torch_dtype=torch.float16,
        attn_implementation="eager", 
        trust_remote_code=True
    ).eval()

    vocab_size = model.config.vocab_size
    print(f"Generating {num_samples} Random Token sequences...")
    
    collected_hiddens = []

    def hook_fn(module, args):
        collected_hiddens.append(args[0].detach().cpu().float())

    target_module = model.model.layers[layer_idx].self_attn.o_proj
    handle = target_module.register_forward_pre_hook(hook_fn)

    batch_size = 20
    num_batches = (num_samples + batch_size - 1) // batch_size
    
    print(f" > Running inference on Random Inputs (Layer {layer_idx})...")
    with torch.no_grad():
        for _ in tqdm(range(num_batches)):
            inp = torch.randint(0, vocab_size, (batch_size, seq_len)).to(device)
            model(inp)
            
    handle.remove()
    del model, tokenizer
    torch.cuda.empty_cache()
    gc.collect()

    all_states = torch.cat(collected_hiddens, dim=0)
    all_states = all_states[:num_samples]
    
    print(f"Calculating statistics on shape: {all_states.shape}...")
    
    std_per_pos_dim = all_states.std(dim=0) # [Seq_Len, Hidden_Dim]
    
    avg_std_per_pos = std_per_pos_dim.mean(dim=-1).numpy() # [Seq_Len]
    
    return avg_std_per_pos

# =========================================
# =========================================

def plot_std_aligned(config, std_data):
    name = config['name']
    layer = config['target_layer']
    color = config['color']

    fig, ax = plt.subplots(figsize=(14, 8), constrained_layout=True)
    
    positions = np.arange(len(std_data))
    
    ax.plot(positions, std_data, 
            color=color, 
            marker='o',
            markeredgecolor='white',
            markeredgewidth=1.5,
            markersize=10,
            linestyle='-',
            linewidth=3,
            alpha=0.9, 
            label='Dimension-wise Std Dev')
    
    ax.set_title(f"[{name}] Dimension-wise Variance Decay\n(Layer {layer}, Random Inputs)", 
                 fontsize=24, pad=20, fontweight='bold')
    ax.set_xlabel("Token Position Index", fontsize=22)
    ax.set_ylabel("Avg. Dimension-wise Std", fontsize=22)
    
    ax.grid(True, linestyle='--', alpha=0.5)
    
    first_val = std_data[0]
    ax.annotate(f"No Aggregation\n(Std: {first_val:.2f})", 
                xy=(0, first_val), 
                xytext=(5, first_val + (first_val * 0.05)),
                arrowprops=dict(facecolor='black', arrowstyle='->', lw=2.5),
                fontsize=20, fontweight='bold')

    ax.set_xlim(-2, len(std_data) + 2)
    
    ax.legend(loc='upper right', frameon=True, fontsize=16)
    
    print("[Plot] Showing aligned figure...")
    plt.show()

if __name__ == "__main__":
    std_data = get_random_input_dim_std(CONFIG, NUM_SAMPLES, SEQ_LEN, DEVICE)
    
    plot_std_aligned(CONFIG, std_data)